phase 1

In [11]:
from pathlib import Path
import pandas as pd
import re
import unicodedata
from sklearn.feature_extraction import _stop_words as sklearn_stopwords

DO_STEM = False 

INPUT_PATH = Path("ecommerceDataset.csv")
OUTPUT_PATH = Path("ecommerce_preprocessed.csv")

assert INPUT_PATH.exists(), f"Input dataset not found at {INPUT_PATH}"

# load once and inspect header problems
df = pd.read_csv(INPUT_PATH, low_memory=False)

# detect suspicious header (long column name => likely no header)
long_name_found = any(isinstance(c, str) and len(c) > 80 and " " in c for c in df.columns)
if long_name_found or df.shape[1] == 1:
    df = pd.read_csv(INPUT_PATH, header=None, low_memory=False)
    if df.shape[1] == 2:
        df.columns = ["category", "text"]
    elif df.shape[1] == 1:
        # fallback: keep as 'text'
        df.columns = ["text"]
    else:
        # assign heuristics for names
        names = []
        for i in range(df.shape[1]):
            col0 = str(df.iloc[0, i])
            if len(col0.split()) <= 3 and all(len(tok) < 20 for tok in col0.split()):
                names.append("category")
            else:
                names.append(f"text_{i}" if i>0 else "category")
        for i, nm in enumerate(names):
            if nm in names[:i]:
                names[i] = f"{nm}_{i}"
        df.columns = names

# heuristics to pick text & label columns
text_candidates = [c for c in df.columns if any(k in c.lower() for k in ("description", "desc", "text", "title", "name", "product"))]
label_candidates = [c for c in df.columns if any(k in c.lower() for k in ("category", "label", "class", "cat"))]

text_col = text_candidates[0] if text_candidates else None
label_col = label_candidates[0] if label_candidates else None

if text_col is None:
    obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if obj_cols:
        text_col = obj_cols[-1]
    else:
        raise ValueError("No text-like column found. Please provide the column name.")

# precompile regexes
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
NON_ALPHANUM_RE = re.compile(r"[^a-zA-Z0-9\s]")
MULTI_SPACE_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"\b\w+\b")

EN_STOPWORDS = set(getattr(sklearn_stopwords, "ENGLISH_STOP_WORDS", []))

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = HTML_TAG_RE.sub(" ", s)
    s = URL_RE.sub(" ", s)
    s = EMAIL_RE.sub(" ", s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join([c for c in s if not unicodedata.combining(c)])
    s = NON_ALPHANUM_RE.sub(" ", s)
    s = MULTI_SPACE_RE.sub(" ", s).strip()
    return s

def tokenize_no_stem(s):
    tokens = WORD_RE.findall(s)
    tokens = [t for t in tokens if t not in EN_STOPWORDS and len(t) > 1]
    return tokens

def preprocess_text_fast(s):
    s2 = normalize_text(s)
    tokens = tokenize_no_stem(s2)
    return " ".join(tokens)

# apply preprocessing
texts = df[text_col].astype(str).tolist()
df["clean_text"] = [preprocess_text_fast(t) for t in texts]

# drop empty and duplicates
df = df[df["clean_text"].str.strip() != ""].copy()
df = df.drop_duplicates(subset=["clean_text"]).reset_index(drop=True)

# if label column exists, drop missing labels
if label_col is not None and label_col in df.columns:
    df = df[~df[label_col].isna()]

# save
df.to_csv(OUTPUT_PATH, index=False)
print("Saved preprocessed dataset to:", OUTPUT_PATH)


Saved preprocessed dataset to: ecommerce_preprocessed.csv


phase 2

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import math
import numpy as np
import pandas as pd
import time
import pickle

# ---------------- CONFIG ----------------
INPUT_CSV = Path("ecommerce_preprocessed.csv")
OUTPUT_DIR = Path("phase2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TF-IDF params
TFIDF_MAX_VOCAB = 10000      # None or int
TFIDF_MIN_DF = 2             # minimum document frequency to keep a term
TFIDF_NORM = "l2"            # "l2" or "none"

# Word2Vec params
W2V_USE_GENSIM = True        # try gensim if available; if not, fallback to PPMI+SVD
W2V_VECTOR_SIZE = 100
W2V_WINDOW = 5
W2V_MIN_COUNT = 2
W2V_MAX_VOCAB = 5000        # if using PPMI+SVD, limit vocab to this for memory

# Save options
SAVE_TFIDF_AS_SPARSE = True   # if scipy is installed will save CSR, otherwise numpy dense
# ----------------------------------------

# Helper: load dataset
def load_dataset(path):
    assert path.exists(), f"Input file not found: {path}"
    df = pd.read_csv(path, low_memory=False)
    if "clean_text" not in df.columns:
        raise ValueError("Expected column 'clean_text' in preprocessed CSV.")
    # label may be 'category' or others
    label_col = "category" if "category" in df.columns else None
    return df, label_col


TF-IDF

In [ ]:
# ---------------- TF-IDF (from scratch) ----------------
def build_vocab_and_df(docs, max_vocab=None, min_df=1):
    """docs: list of strings (tokens separated by spaces)"""
    df_counts = Counter()
    tf_counters = []
    total_docs = len(docs)
    for doc in docs:
        tokens = doc.split()
        # unique tokens in doc for df
        seen = set()
        for t in tokens:
            if t:
                if t not in seen:
                    df_counts[t] += 1
                    seen.add(t)
    # filter by min_df
    items = [(w, c) for w, c in df_counts.items() if c >= min_df]
    # sort by df desc (or frequency)
    items.sort(key=lambda x: (-x[1], x[0]))
    if max_vocab:
        items = items[:max_vocab]
    vocab = {w: idx for idx, (w, _) in enumerate(items)}
    df_final = {w: c for w, c in items}
    return vocab, df_final

def docs_to_tf_counts(docs, vocab):
    """Return list of dicts: each doc as {idx: count}"""
    res = []
    for doc in docs:
        cnt = Counter()
        for t in doc.split():
            if t in vocab:
                cnt[vocab[t]] += 1
        res.append(dict(cnt))
    return res

def compute_idf(df_counts, n_docs, smooth=True):
    """df_counts: dict term->df_count"""
    idf = {}
    for term, dfc in df_counts.items():
        if smooth:
            idf_val = math.log((n_docs + 1) / (dfc + 1)) + 1.0
        else:
            idf_val = math.log(n_docs / (dfc)) if dfc>0 else 0.0
        idf[term] = idf_val
    return idf

def build_tfidf_matrix(doc_tf_dicts, vocab, idf_map, norm="l2", use_scipy_if_possible=True):
    """
    Returns:
      - X: scipy.sparse.csr_matrix if scipy available and requested, otherwise numpy array (dense)
      - row_doc_indices, col_indices, data if sparse-like and saved separately
    """
    try:
        if use_scipy_if_possible:
            import scipy.sparse as sp
            SCIPY_AVAILABLE = True
        else:
            SCIPY_AVAILABLE = False
    except Exception:
        SCIPY_AVAILABLE = False

    n_docs = len(doc_tf_dicts)
    V = len(vocab)
    if SCIPY_AVAILABLE:
        # build lists for sparse matrix
        rows = []
        cols = []
        data = []
        for i, doc in enumerate(doc_tf_dicts):
            # compute tf (normalized)
            total_terms = sum(doc.values())
            for idx, cnt in doc.items():
                tf = cnt / total_terms if total_terms > 0 else 0.0
                # lookup term by index -> need reverse mapping for idf
                # we stored idf_map keyed by token string; build idf_by_index first
                rows.append(i)
                cols.append(idx)
                # placeholder for tf*idf; we'll compute idf_by_index separately
                data.append(tf)
        # convert to CSR then multiply columns by idf vector
        mat = sp.csr_matrix((data, (rows, cols)), shape=(n_docs, V), dtype=np.float64)
        # build idf_by_index
        idf_by_index = np.zeros(V, dtype=np.float64)
        # idf_map keys are tokens; need index of each token string
        # We assume vocab: token -> idx
        for token, idx in vocab.items():
            idf_val = idf_map.get(token, 0.0)
            idf_by_index[idx] = idf_val
        # multiply columns by idf
        mat = mat.multiply(idf_by_index)
        # normalize rows if requested
        if norm == "l2":
            from sklearn.preprocessing import normalize
            mat = normalize(mat, norm='l2', axis=1)
        return mat
    else:
        # dense fallback (may be memory heavy)
        X = np.zeros((n_docs, V), dtype=np.float32)
        # precompute idf_by_index
        idf_by_index = np.zeros(V, dtype=np.float32)
        for token, idx in vocab.items():
            idf_by_index[idx] = float(idf_map.get(token, 0.0))
        for i, doc in enumerate(doc_tf_dicts):
            total_terms = sum(doc.values())
            if total_terms == 0:
                continue
            for idx, cnt in doc.items():
                tf = cnt / total_terms
                X[i, idx] = tf * idf_by_index[idx]
        if norm == "l2":
            norms = np.linalg.norm(X, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            X = X / norms
        return X

Word2Vec

In [ ]:
# ---------------- Word2Vec (gensim or PPMI+SVD fallback) ----------------
def train_word2vec_gensim(docs_tokens, vector_size=100, window=5, min_count=2, save_path=None):
    try:
        from gensim.models import Word2Vec
    except Exception as e:
        print("gensim not available:", e)
        return None, None
    sentences = [doc.split() for doc in docs_tokens]
    model = Word2Vec(sentences, vector_size=vector_size, window=window, min_count=min_count, workers=4, epochs=5)
    w2v_vocab = {w: model.wv[w] for w in model.wv.index_to_key}
    if save_path:
        model.save(str(save_path))
    return model, w2v_vocab

def build_cooccurrence_matrix(docs_tokens, vocab, window=5):
    """
    Build symmetric co-occurrence counts matrix for vocab indices.
    vocab: token->idx (we typically limit vocab size)
    returns: cooc (V,V) numpy float64, word_counts (V,)
    """
    V = len(vocab)
    cooc = np.zeros((V, V), dtype=np.float64)
    word_counts = np.zeros(V, dtype=np.float64)
    for doc in docs_tokens:
        tokens = doc.split()
        idxs = [vocab[t] for t in tokens if t in vocab]
        L = len(idxs)
        for i, center in enumerate(idxs):
            word_counts[center] += 1
            left = max(0, i - window)
            right = min(L, i + window + 1)
            for j in range(left, right):
                if j == i: continue
                context = idxs[j]
                cooc[center, context] += 1.0
    return cooc, word_counts

def ppmi_from_cooc(cooc, word_counts, positive=True):
    """
    Compute PPMI matrix from co-occurrence counts.
    cooc: VxV co-occurrence counts
    word_counts: array length V (counts of word as center)
    """
    total_cooc = cooc.sum()
    V = cooc.shape[0]
    # To avoid divide-by-zero, add tiny eps
    eps = 1e-8
    col_sums = cooc.sum(axis=0) + eps  # context totals
    row_sums = cooc.sum(axis=1) + eps  # target totals
    ppmi = np.zeros_like(cooc)
    for i in range(V):
        for j in range(V):
            nij = cooc[i, j]
            if nij <= 0:
                continue
            p_ij = nij / total_cooc
            p_i = row_sums[i] / total_cooc
            p_j = col_sums[j] / total_cooc
            val = math.log(p_ij / (p_i * p_j) + 1e-10)
            if positive and val < 0:
                val = 0.0
            ppmi[i, j] = val
    return ppmi

def svd_embeddings_from_matrix(M, dim=100):
    """
    Compute SVD and return embeddings of shape (V, dim).
    Use U * sqrt(S) trick for symmetric matrices.
    """
    print("Performing SVD on matrix of shape", M.shape)
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    # take top-dim
    U_d = U[:, :dim]
    S_d = S[:dim]
    # embeddings: U_d * sqrt(S_d)
    E = U_d * np.sqrt(S_d[np.newaxis, :])
    return E

# ---------------- Utilities: document vector from word vectors ----------------
def doc_vector_by_mean(doc_texts, w2v_vocab_indexable):
    """
    doc_texts: list of strings (tokens separated by spaces)
    w2v_vocab_indexable: dict token->vector (numpy array)
    returns: numpy array (n_docs, vec_dim)
    """
    n = len(doc_texts)
    sample_token = next(iter(w2v_vocab_indexable))
    dim = w2v_vocab_indexable[sample_token].shape[0]
    X = np.zeros((n, dim), dtype=np.float32)
    for i, doc in enumerate(doc_texts):
        toks = [t for t in doc.split() if t in w2v_vocab_indexable]
        if len(toks) == 0:
            continue
        vecs = np.vstack([w2v_vocab_indexable[t] for t in toks])
        mean = vecs.mean(axis=0)
        X[i] = mean
    return X


In [ ]:
# ---------------- Main pipeline ----------------
def main():
    print("Loading dataset...")
    df, label_col = load_dataset(INPUT_CSV)
    docs = df["clean_text"].astype(str).tolist()
    labels = df[label_col].tolist() if label_col else None
    n_docs = len(docs)
    print(f"Loaded {n_docs} documents.")

    # ---- TF-IDF ----
    print("\nBuilding TF-IDF vocabulary...")
    vocab, df_counts = build_vocab_and_df(docs, max_vocab=TFIDF_MAX_VOCAB, min_df=TFIDF_MIN_DF)
    print(f"TF-IDF vocab size: {len(vocab)} (min_df={TFIDF_MIN_DF})")
    doc_tf = docs_to_tf_counts(docs, vocab)
    idf_map = compute_idf({w: df_counts[w] for w in df_counts}, n_docs)
    print("Computing TF-IDF matrix...")
    X_tfidf = build_tfidf_matrix(doc_tf, vocab, idf_map, norm=TFIDF_NORM, use_scipy_if_possible=SAVE_TFIDF_AS_SPARSE)
    # Save tfidf and vocab
    vocab_path = OUTPUT_DIR / "tfidf_vocab.json"
    with open(vocab_path, "w", encoding="utf8") as f:
        json.dump(vocab, f, ensure_ascii=False)
    print("Saved TF-IDF vocab ->", vocab_path)

    if hasattr(X_tfidf, "toarray") or str(type(X_tfidf)).find("csr") != -1:
        # scipy sparse
        try:
            import scipy.sparse as sp
            sp.save_npz(str(OUTPUT_DIR / "X_tfidf.npz"), X_tfidf)
            print("Saved TF-IDF sparse matrix ->", OUTPUT_DIR / "X_tfidf.npz")
        except Exception as e:
            print("Could not save sparse via scipy:", e)
            # fallback to dense
            dense = X_tfidf.toarray() if hasattr(X_tfidf, "toarray") else np.array(X_tfidf)
            np.save(OUTPUT_DIR / "X_tfidf.npy", dense)
            print("Saved TF-IDF dense matrix ->", OUTPUT_DIR / "X_tfidf.npy")
    else:
        # dense numpy
        np.save(OUTPUT_DIR / "X_tfidf.npy", X_tfidf)
        print("Saved TF-IDF dense matrix ->", OUTPUT_DIR / "X_tfidf.npy")

    # Save idf map
    with open(OUTPUT_DIR / "idf_map.pkl", "wb") as f:
        pickle.dump(idf_map, f)
    print("Saved idf_map ->", OUTPUT_DIR / "idf_map.pkl")

    # ---- Word2Vec ----
    print("\nTraining Word2Vec (or fallback)...")
    start = time.time()
    model = None
    w2v_vocab = None
    if W2V_USE_GENSIM:
        try:
            model, w2v_vocab = train_word2vec_gensim(docs, vector_size=W2V_VECTOR_SIZE,
                                                     window=W2V_WINDOW, min_count=W2V_MIN_COUNT,
                                                     save_path=OUTPUT_DIR / "gensim_w2v.model")
            if model is not None:
                print("Trained gensim Word2Vec, vocab size:", len(w2v_vocab))
        except Exception as e:
            print("gensim training failed or not available, fallback:", e)
            model = None
    if model is None:
        # fallback: PPMI + SVD on top frequent words
        print("Using PPMI+SVD fallback. Building limited vocab...")
        # build vocab by term frequency across corpus
        all_tokens = Counter()
        for d in docs:
            all_tokens.update(d.split())
        most_common = all_tokens.most_common(W2V_MAX_VOCAB)
        wvocab = {w: idx for idx, (w, _) in enumerate(most_common)}
        print(f"PPMI vocab size = {len(wvocab)} (max {W2V_MAX_VOCAB})")
        cooc, wcounts = build_cooccurrence_matrix(docs, wvocab, window=W2V_WINDOW)
        ppmi = ppmi_from_cooc(cooc, wcounts, positive=True)
        emb = svd_embeddings_from_matrix(ppmi, dim=W2V_VECTOR_SIZE)
        # embeddings dict
        w2v_vocab = {w: emb[idx] for w, idx in wvocab.items()}
        # save small mapping
        np.save(OUTPUT_DIR / "w2v_ppmi_embeddings.npy", emb)
        with open(OUTPUT_DIR / "w2v_ppmi_vocab.json", "w", encoding="utf8") as f:
            json.dump({w: int(idx) for w, idx in wvocab.items()}, f, ensure_ascii=False)
        print("Saved PPMI+SVD embeddings and vocab.")

    # if gensim model exists, extract vectors into dict
    if model is not None and w2v_vocab is None:
        w2v_vocab = {w: model.wv[w] for w in model.wv.index_to_key}
        # optionally save vectors in numpy
        vocab_list = model.wv.index_to_key
        mat = np.vstack([model.wv[w] for w in vocab_list])
        np.save(OUTPUT_DIR / "w2v_gensim_embeddings.npy", mat)
        with open(OUTPUT_DIR / "w2v_gensim_vocab.json", "w", encoding="utf8") as f:
            json.dump(vocab_list, f, ensure_ascii=False)
        print("Saved gensim embeddings and vocab.")

    # make document embeddings by averaging word embeddings
    print("Building document-level Word2Vec vectors (mean of token vectors)...")
    X_w2v = doc_vector_by_mean(docs, w2v_vocab)
    np.save(OUTPUT_DIR / "X_w2v_mean.npy", X_w2v)
    print("Saved X_w2v_mean ->", OUTPUT_DIR / "X_w2v_mean.npy")

    # Save labels and indices
    if labels is not None:
        np.save(OUTPUT_DIR / "labels.npy", np.array(labels))
        print("Saved labels ->", OUTPUT_DIR / "labels.npy")

    # Summarize
    elapsed = time.time() - start
    print(f"\nPhase 2 complete. Outputs in: {OUTPUT_DIR}. Time elapsed: {elapsed:.1f}s")

if __name__ == "__main__":
    main()


Loading dataset...
Loaded 27578 documents.

Building TF-IDF vocabulary...
TF-IDF vocab size: 10000 (min_df=2)
Computing TF-IDF matrix...
Saved TF-IDF vocab -> phase2_outputs\tfidf_vocab.json
Saved TF-IDF sparse matrix -> phase2_outputs\X_tfidf.npz
Saved idf_map -> phase2_outputs\idf_map.pkl

Training Word2Vec (or fallback)...
Trained gensim Word2Vec, vocab size: 44020
Building document-level Word2Vec vectors (mean of token vectors)...
Saved X_w2v_mean -> phase2_outputs\X_w2v_mean.npy
Saved labels -> phase2_outputs\labels.npy

Phase 2 complete. Outputs in: phase2_outputs. Time elapsed: 10.4s


phase 3

initialization

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import time

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt

# Config
PHASE2_DIR = Path("phase2_outputs")
PREP_CSV = Path("ecommerce_preprocessed.csv")  # fallback for labels
OUT_DIR = Path("phase3_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.2
TSNE_SAMPLE = 3000  # sample size for t-SNE if dataset large (to limit time)
SAVE_MODELS = True
N_JOBS = -1

# --- Utilities for loading ---
def load_tfidf_matrix(path):
    """Try .npz (sparse) then .npy (dense)"""
    try:
        import scipy.sparse as sp
    except Exception:
        sp = None
    p_npz = path / "X_tfidf.npz"
    p_npy = path / "X_tfidf.npy"
    if p_npz.exists() and sp is not None:
        mat = sp.load_npz(str(p_npz))
        return mat
    elif p_npy.exists():
        return np.load(str(p_npy), mmap_mode=None)
    else:
        print("TF-IDF matrix not found in", path)
        return None

def load_w2v(path):
    p = path / "X_w2v_mean.npy"
    if p.exists():
        return np.load(str(p))
    else:
        print("W2V doc matrix not found in", p)
        return None

def load_labels(path):
    p = path / "labels.npy"
    if p.exists():
        return np.load(str(p))
    # fallback: try to read from preprocessed csv
    if PREP_CSV.exists():
        df = pd.read_csv(PREP_CSV, low_memory=False)
        if "category" in df.columns:
            return df["category"].astype(str).values
    raise FileNotFoundError("Labels not found in phase2 outputs or preprocessed CSV.")


In [ ]:
def eval_and_report(y_true, y_pred, labels, out_prefix):
    acc = accuracy_score(y_true, y_pred)
    # average='weighted' => precision, recall, f1 به صورت اسکالر برمی‌گردند و support = None
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    report = classification_report(y_true, y_pred, zero_division=0)
    # confusion matrix باید با لیستِ برچسب‌ها ساخته شود
    if labels is None:
        labels = np.unique(np.concatenate([y_true, y_pred]))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    # save classification report
    with open(f"{out_prefix}_classification_report.txt", "w", encoding="utf8") as f:
        f.write(report)
    # save confusion matrix csv
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.to_csv(f"{out_prefix}_confusion_matrix.csv")
    # total support (تعداد نمونه‌های تست) — امن و دقیق
    support_total = int(len(y_true))
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "support_total": support_total}


def plot_confusion_matrix(cm_df, out_png, figsize=(8,8)):
    try:
        import seaborn as sns
        plt.figure(figsize=figsize)
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.ylabel("True")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(out_png)
        plt.close()
    except Exception:
        # fallback minimal plot
        plt.figure(figsize=figsize)
        plt.imshow(cm_df.values, interpolation='nearest', cmap=plt.cm.Blues)
        plt.colorbar()
        ticks = range(len(cm_df))
        plt.xticks(ticks, cm_df.columns, rotation=45, ha='right')
        plt.yticks(ticks, cm_df.index)
        plt.ylabel("True")
        plt.xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(out_png)
        plt.close()

# --- Visualization helpers ---
def plot_2d_embeddings(X, y, labels, out_png, method="PCA", sample_size=2000, random_state=RANDOM_STATE):
    """
    X: (n_samples, dim) dense numpy
    y: labels array
    labels: unique label list (for color mapping)
    """
    n = X.shape[0]
    if n > sample_size:
        np.random.seed(random_state)
        idx = np.random.choice(n, sample_size, replace=False)
        Xs = X[idx]
        ys = y[idx]
    else:
        Xs = X
        ys = y
    if method == "PCA":
        red = PCA(n_components=2, random_state=random_state).fit_transform(Xs)
    else:
        red = TSNE(n_components=2, random_state=random_state, perplexity=30, n_iter=800).fit_transform(Xs)
    plt.figure(figsize=(8,6))
    unique = list(labels)
    cmap = plt.get_cmap("tab10")
    for i, lab in enumerate(unique):
        mask = (ys == lab)
        if mask.sum() == 0:
            continue
        plt.scatter(red[mask,0], red[mask,1], label=str(lab), s=8, alpha=0.7)
    plt.legend(markerscale=2, bbox_to_anchor=(1.05,1), loc='upper left')
    plt.title(f"{method} visualization")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()


the main part of phase 3

In [ ]:
# --- Main pipeline ---
def main():
    print("Loading matrices and labels...")
    X_tfidf = load_tfidf_matrix(PHASE2_DIR)
    X_w2v = load_w2v(PHASE2_DIR)
    labels = load_labels(PHASE2_DIR)
    labels = np.array(labels)
    unique_labels = np.unique(labels).tolist()
    print("Documents:", len(labels), "| Classes:", len(unique_labels))

    # Convert sparse TF-IDF to array only if necessary for some classifiers
    is_tfidf_sparse = False
    try:
        import scipy.sparse as sp
        is_tfidf_sparse = sp.issparse(X_tfidf)
    except Exception:
        is_tfidf_sparse = False

    # Prepare dataset splits: use same train/test indices for both representations
    idxs = np.arange(len(labels))
    X_idx_train, X_idx_test, y_train, y_test = train_test_split(idxs, labels, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=labels)
    print("Train docs:", len(X_idx_train), "| Test docs:", len(X_idx_test))

    results = []
    models_to_run = {
        "LogisticRegression": LogisticRegression(max_iter=2000, C=1.0, n_jobs=N_JOBS, random_state=RANDOM_STATE),
        "LinearSVC": LinearSVC(max_iter=20000, dual=False, random_state=RANDOM_STATE),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=N_JOBS),
    }

    # For each representation, train & eval each model
    for repr_name, X_repr in [("TFIDF", X_tfidf), ("W2V_mean", X_w2v)]:
        if X_repr is None:
            print(f"Skipping {repr_name} because matrix not found.")
            continue

        print("\n=== Representation:", repr_name, "===")
        # prepare X_train and X_test
        if repr_name == "TFIDF" and is_tfidf_sparse:
            # slicing sparse by index arrays
            X_train = X_repr[X_idx_train]
            X_test = X_repr[X_idx_test]
        else:
            # ensure dense numpy arrays
            X_all = np.array(X_repr)
            X_train = X_all[X_idx_train]
            X_test = X_all[X_idx_test]

        # If representation is dense and not scaled, scale for models that need it
        need_scaling = (repr_name == "W2V_mean")
        scaler = None
        if need_scaling:
            scaler = StandardScaler(with_mean=True, with_std=True)
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)

        for model_name, model in models_to_run.items():
            print(f"Training {model_name} on {repr_name} ...")
            t0 = time.time()
            clf = None
            try:
                clf = model
                clf.fit(X_train, y_train)
            except Exception as e:
                print("Training failed for", model_name, "on", repr_name, ":", e)
                continue
            t1 = time.time()
            y_pred = clf.predict(X_test)
            metrics = eval_and_report(y_test, y_pred, labels=unique_labels, out_prefix=str(OUT_DIR / f"{repr_name}_{model_name}"))
            metrics.update({"repr": repr_name, "model": model_name, "train_time_s": t1-t0})
            results.append(metrics)
            # save confusion matrix png
            cm_df = pd.read_csv(str(OUT_DIR / f"{repr_name}_{model_name}_confusion_matrix.csv"), index_col=0)
            plot_confusion_matrix(cm_df, str(OUT_DIR / f"{repr_name}_{model_name}_confusion_matrix.png"))
            print(f"Saved confusion matrix plot for {repr_name}_{model_name}")

            # save model (including scaler if any)
            if SAVE_MODELS:
                save_p = OUT_DIR / f"{repr_name}_{model_name}_model.pkl"
                to_save = {"model": clf}
                if scaler is not None:
                    to_save["scaler"] = scaler
                with open(save_p, "wb") as f:
                    pickle.dump(to_save, f)
                print("Saved model to", save_p)

            # save a few example documents per predicted class from test set
            examples_p = OUT_DIR / f"{repr_name}_{model_name}_examples_per_class.txt"
            with open(examples_p, "w", encoding="utf8") as f:
                f.write(f"Examples for model {model_name} on representation {repr_name}\n\n")
                # map test idx -> predicted
                for cls in unique_labels:
                    f.write(f"=== CLASS: {cls} ===\n")
                    # find indices in test set that predicted this class
                    sel = np.where(y_pred == cls)[0]
                    if len(sel) == 0:
                        f.write("NO PREDICTIONS\n\n")
                        continue
                    # pick up to 5 examples
                    sel_idx = sel[:5]
                    for si in sel_idx:
                        doc_idx = X_idx_test[si]
                        # load original text from preprocessed csv as sample (fallback)
                        try:
                            df = pd.read_csv(PREP_CSV, low_memory=False)
                            text = df.iloc[doc_idx]["clean_text"]
                        except Exception:
                            text = "[original text not available]"
                        f.write(f"- doc_idx {doc_idx}: {str(text)[:400]}\n")
                    f.write("\n")
            print("Saved example documents file:", examples_p)

    # Save results table
    results_df = pd.DataFrame(results)
    results_df = results_df[["repr", "model", "accuracy", "precision", "recall", "f1", "train_time_s", "support_total"]]
    results_df.to_csv(OUT_DIR / "classification_results_summary.csv", index=False)
    print("Saved summary results to", OUT_DIR / "classification_results_summary.csv")

    # Visualizations: PCA and t-SNE for each repr (use sampled subset for t-SNE if needed)
    print("\nGenerating visualizations (PCA + t-SNE sample)...")
    if X_tfidf is not None:
        # convert to dense for PCA/t-SNE but sample to reasonable size
        try:
            if is_tfidf_sparse:
                X_dense = X_tfidf.toarray()
            else:
                X_dense = np.array(X_tfidf)
            plot_2d_embeddings(X_dense, labels, unique_labels, str(OUT_DIR / "tfidf_pca.png"), method="PCA", sample_size=TSNE_SAMPLE)
            # t-SNE (sampled)
            plot_2d_embeddings(X_dense, labels, unique_labels, str(OUT_DIR / "tfidf_tsne.png"), method="TSNE", sample_size=TSNE_SAMPLE)
            print("Saved TFIDF PCA/t-SNE plots.")
        except Exception as e:
            print("Could not visualize TFIDF:", e)

    if X_w2v is not None:
        try:
            X_dense = np.array(X_w2v)
            plot_2d_embeddings(X_dense, labels, unique_labels, str(OUT_DIR / "w2v_pca.png"), method="PCA", sample_size=TSNE_SAMPLE)
            plot_2d_embeddings(X_dense, labels, unique_labels, str(OUT_DIR / "w2v_tsne.png"), method="TSNE", sample_size=TSNE_SAMPLE)
            print("Saved W2V PCA/t-SNE plots.")
        except Exception as e:
            print("Could not visualize W2V:", e)

    print("\nPhase 3 complete. Outputs are in:", OUT_DIR)

if __name__ == "__main__":
    main()


Loading matrices and labels...
Documents: 27578 | Classes: 4
Train docs: 22062 | Test docs: 5516

=== Representation: TFIDF ===
Training LogisticRegression on TFIDF ...
Saved confusion matrix plot for TFIDF_LogisticRegression
Saved model to phase3_outputs\TFIDF_LogisticRegression_model.pkl
Saved example documents file: phase3_outputs\TFIDF_LogisticRegression_examples_per_class.txt
Training LinearSVC on TFIDF ...
Saved confusion matrix plot for TFIDF_LinearSVC
Saved model to phase3_outputs\TFIDF_LinearSVC_model.pkl
Saved example documents file: phase3_outputs\TFIDF_LinearSVC_examples_per_class.txt
Training RandomForest on TFIDF ...
Saved confusion matrix plot for TFIDF_RandomForest
Saved model to phase3_outputs\TFIDF_RandomForest_model.pkl
Saved example documents file: phase3_outputs\TFIDF_RandomForest_examples_per_class.txt

=== Representation: W2V_mean ===
Training LogisticRegression on W2V_mean ...
Saved confusion matrix plot for W2V_mean_LogisticRegression
Saved model to phase3_outp

phase 4

In [ ]:
import os
from pathlib import Path
import pickle
import json
import numpy as np
import pandas as pd
import math
import time
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, roc_curve, auc, precision_recall_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.utils import resample

# Config (match phase3)
PHASE2_DIR = Path("phase2_outputs")
PHASE3_DIR = Path("phase3_outputs")
PREP_CSV = Path("ecommerce_preprocessed.csv")
OUT_DIR = Path("phase4_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.2
RANDOM_STATE = 42
BOOTSTRAP_ITERS = 1000
BOOTSTRAP_SAMPLE_RATIO = 1.0  # sample with replacement size = len(test) * ratio

# util loaders
def load_labels():
    p = PHASE2_DIR / "labels.npy"
    if p.exists():
        return np.load(str(p))
    if PREP_CSV.exists():
        df = pd.read_csv(PREP_CSV, low_memory=False)
        if "category" in df.columns:
            return df["category"].astype(str).values
    raise FileNotFoundError("Labels not found in phase2 outputs or preprocessed CSV.")

def load_repr_matrix(repr_name):
    # repr_name: "TFIDF" or "W2V_mean"
    if repr_name == "TFIDF":
        # try npz then npy
        p_npz = PHASE2_DIR / "X_tfidf.npz"
        p_npy = PHASE2_DIR / "X_tfidf.npy"
        if p_npz.exists():
            import scipy.sparse as sp
            return sp.load_npz(str(p_npz))
        elif p_npy.exists():
            return np.load(str(p_npy))
        else:
            return None
    elif repr_name == "W2V_mean":
        p = PHASE2_DIR / "X_w2v_mean.npy"
        if p.exists():
            return np.load(str(p))
        else:
            return None
    else:
        return None

def list_saved_models():
    # find all *_model.pkl files in phase3 outputs
    return sorted(PHASE3_DIR.glob("*_model.pkl"))


In [ ]:
# evaluation helpers
def safe_predict_proba_or_score(model_obj, X):
    """
    Try to get probability scores suitable for multiclass ROC/PR.
    Returns: (has_scores_bool, scores_array_or_None)
    scores_array shape: (n_samples, n_classes) or (n_samples,) for binary.
    """
    try:
        if hasattr(model_obj, "predict_proba"):
            probs = model_obj.predict_proba(X)
            return True, probs
        elif hasattr(model_obj, "decision_function"):
            dec = model_obj.decision_function(X)
            # decision_function may return (n_samples, n_classes) or (n_samples,)
            if dec.ndim == 1:
                # binary — convert to 2-column by [-dec, dec] trick
                dec2 = np.vstack([-dec, dec]).T
                return True, dec2
            return True, dec
    except Exception as e:
        print("score extraction failed:", e)
    return False, None

def plot_and_save_confusion(cm, labels, out_png):
    plt.figure(figsize=(8,6))
    try:
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
        plt.xlabel("Predicted"); plt.ylabel("True")
    except Exception:
        plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
        plt.colorbar()
        ticks = range(len(labels))
        plt.xticks(ticks, labels, rotation=45, ha='right')
        plt.yticks(ticks, labels)
        plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

def save_classification_metrics(y_true, y_pred, labels, prefix):
    # overall
    acc = accuracy_score(y_true, y_pred)
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    # per-class
    precs, recs, f1s, supports = precision_recall_fscore_support(y_true, y_pred, average=None, labels=labels, zero_division=0)
    # confusion
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    # save report
    report_text = classification_report(y_true, y_pred, labels=labels, zero_division=0)
    with open(f"{prefix}_report.txt", "w", encoding="utf8") as f:
        f.write(report_text)
    # save summary json
    summary = {
        "accuracy": float(acc),
        "precision_weighted": float(precision_w),
        "recall_weighted": float(recall_w),
        "f1_weighted": float(f1_w),
        "precision_macro": float(precision_m),
        "recall_macro": float(recall_m),
        "f1_macro": float(f1_m),
        "per_class": {str(l): {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(s)} for l,p,r,f,s in zip(labels, precs, recs, f1s, supports)}
    }
    with open(f"{prefix}_summary.json", "w", encoding="utf8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    # save confusion csv and plot
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    cm_df.to_csv(f"{prefix}_confusion.csv")
    plot_and_save_confusion(cm, labels, f"{prefix}_confusion.png")
    return summary, cm, cm_df


In [ ]:
def compute_roc_pr(y_true, y_score, labels, out_prefix):
    # y_score shape: (n_samples, n_classes)
    n_classes = len(labels)
    # binarize
    y_bin = label_binarize(y_true, classes=labels)
    results = {}
    for i, lab in enumerate(labels):
        if y_score is None:
            break
        try:
            scores_i = y_score[:, i]
            fpr, tpr, _ = roc_curve(y_bin[:, i], scores_i)
            roc_auc = auc(fpr, tpr)
            precision, recall, _ = precision_recall_curve(y_bin[:, i], scores_i)
            pr_auc = auc(recall, precision)
            # save curves
            plt.figure(figsize=(6,4))
            plt.plot(fpr, tpr, label=f"AUC={roc_auc:.4f}")
            plt.plot([0,1],[0,1],"--", color="gray")
            plt.title(f"ROC for class {lab}")
            plt.xlabel("FPR"); plt.ylabel("TPR")
            plt.legend()
            plt.tight_layout()
            plt.savefig(f"{out_prefix}_roc_class_{i}.png")
            plt.close()
            plt.figure(figsize=(6,4))
            plt.plot(recall, precision, label=f"PR-AUC={pr_auc:.4f}")
            plt.title(f"PR curve for class {lab}")
            plt.xlabel("Recall"); plt.ylabel("Precision")
            plt.legend()
            plt.tight_layout()
            plt.savefig(f"{out_prefix}_pr_class_{i}.png")
            plt.close()
            results[str(lab)] = {"roc_auc": float(roc_auc), "pr_auc": float(pr_auc)}
        except Exception as e:
            results[str(lab)] = {"error": str(e)}
    # save aggregated
    with open(f"{out_prefix}_rocpr.json", "w", encoding="utf8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results

def mcnemar_test(y_true, y_pred_a, y_pred_b):
    """
    McNemar's test (continuity-corrected)
    Returns (statistic, pvalue) if scipy available; otherwise returns (statistic, None)
    """
    # contingency b = A correct, B wrong ; c = A wrong, B correct
    a_correct = (y_pred_a == y_true)
    b_correct = (y_pred_b == y_true)
    b = int(((a_correct) & (~b_correct)).sum())
    c = int(((~a_correct) & (b_correct)).sum())
    stat = ((abs(b - c) - 1)**2) / (b + c) if (b + c) > 0 else 0.0
    pval = None
    try:
        from scipy.stats import chi2
        pval = float(chi2.sf(stat, 1))
    except Exception:
        pval = None
    return {"b": b, "c": c, "chi2_stat": float(stat), "p_value": pval}

def bootstrap_accuracy_diff(y_true, y_pred_a, y_pred_b, iters=1000, seed=42):
    """
    Bootstrap CI for difference in accuracy (acc_A - acc_B).
    Returns dict with observed_diff and lower/upper percentiles (95%).
    """
    np.random.seed(seed)
    n = len(y_true)
    diffs = []
    for _ in range(iters):
        idx = np.random.choice(n, size=n, replace=True)
        accA = accuracy_score(y_true[idx], y_pred_a[idx])
        accB = accuracy_score(y_true[idx], y_pred_b[idx])
        diffs.append(accA - accB)
    diffs = np.array(diffs)
    lo = np.percentile(diffs, 2.5)
    hi = np.percentile(diffs, 97.5)
    obs = accuracy_score(y_true, y_pred_a) - accuracy_score(y_true, y_pred_b)
    return {"observed_diff": float(obs), "ci_lower": float(lo), "ci_upper": float(hi), "bootstrap_samples": int(iters)}


In [ ]:
# --- Main execution ---
def main():
    start_time = time.time()
    labels_arr = load_labels()
    n_docs = len(labels_arr)
    classes = np.unique(labels_arr)
    print("Documents:", n_docs, "Classes:", classes)

    # reproduce indices same as phase3
    idxs = np.arange(n_docs)
    idx_train, idx_test, y_train, y_test = train_test_split(idxs, labels_arr, test_size=TEST_SIZE,
                                                            random_state=RANDOM_STATE, stratify=labels_arr)
    print("Train size:", len(idx_train), "Test size:", len(idx_test))

    # load representations once (we will select later)
    X_tfidf = load_repr_matrix("TFIDF")
    X_w2v = load_repr_matrix("W2V_mean")
    if X_tfidf is None:
        print("TFIDF matrix not found. Some models may fail.")
    if X_w2v is None:
        print("W2V matrix not found.")

    # find saved models
    model_files = list_saved_models()
    if not model_files:
        print("No saved models found in", PHASE3_DIR)
        return
    print("Found model files:", model_files)

    # will store all per-model info
    all_results = {}
    preds_store = {}

    for model_path in model_files:
        name = model_path.stem  # e.g., "TFIDF_LogisticRegression_model" or "W2V_mean_RandomForest_model"
        print("\nProcessing model file:", model_path, "-> name:", name)
        try:
            with open(model_path, "rb") as f:
                saved = pickle.load(f)
        except Exception as e:
            print("Could not load pickle:", e)
            continue

        # saved may be dict with keys 'model' and optionally 'scaler'
        if isinstance(saved, dict) and "model" in saved:
            model = saved["model"]
            scaler = saved.get("scaler", None)
        else:
            # assume the pickle is the model itself
            model = saved
            scaler = None

        # infer representation from filename: it was saved as f"{repr}_{model_name}_model.pkl" in phase3
        parts = name.split("_")
        # restore repr maybe 'TFIDF' or 'W2V' or 'W2V_mean' - attempt to detect
        if parts[0].upper() in ("TFIDF", "TF-IDF"):
            repr_name = "TFIDF"
        elif parts[0].upper().startswith("W2V") or parts[0].upper().startswith("WORD2VEC"):
            repr_name = "W2V_mean"
        else:
            # fallback: try to see if file contains 'tfidf' or 'w2v'
            s = name.lower()
            repr_name = "W2V_mean" if "w2v" in s or "w2v" in model_path.name.lower() else "TFIDF"

        # select X_all accordingly
        if repr_name == "TFIDF":
            X_all = X_tfidf
        else:
            X_all = X_w2v

        if X_all is None:
            print("Representation matrix for", repr_name, "is missing; skipping model.")
            continue

        # prepare X_test selection (handle sparse)
        try:
            import scipy.sparse as sp
            is_sparse = sp.issparse(X_all)
        except Exception:
            is_sparse = False

        if is_sparse:
            X_test = X_all[idx_test]
            X_train_mat = X_all[idx_train]  # not used but could be used if scaler expects fit
        else:
            X_all_arr = np.array(X_all)
            X_test = X_all_arr[idx_test]
            X_train_mat = X_all_arr[idx_train]

        # apply scaler if present
        if scaler is not None:
            # scaler from phase3 was fit on training data; just transform test
            try:
                X_test = scaler.transform(X_test)
            except Exception as e:
                print("Scaler transform failed:", e)

        # predict
        try:
            y_pred = model.predict(X_test)
        except Exception as e:
            print("Prediction failed:", e)
            continue

        # store predictions
        preds_store[name] = {"y_pred": y_pred.tolist(), "repr": repr_name}
        # compute metrics
        prefix = OUT_DIR / name
        summary, cm, cm_df = save_classification_metrics(y_test, y_pred, classes, str(prefix))
        # compute ROC/PR if possible
        has_scores, scores = safe_predict_proba_or_score(model, X_test)
        if has_scores and scores is not None:
            try:
                rocpr = compute_roc_pr(y_test, scores, classes, str(prefix))
            except Exception as e:
                print("ROC/PR computation failed:", e)
                rocpr = None
        else:
            rocpr = None

        summary["repr"] = repr_name
        summary["model_name"] = name
        summary["has_scores"] = bool(has_scores)
        summary["rocpr_summary"] = rocpr
        all_results[name] = summary

        # save misclassified examples for top confusion pairs
        try:
            cm_flat = cm.copy()
            # find top confused pairs (excluding diagonal)
            import numpy as _np
            cm_no_diag = cm_flat.copy()
            np.fill_diagonal(cm_no_diag, 0)
            # get indices of top 3 confusion pairs
            flat = cm_no_diag.flatten()
            top_idxs = flat.argsort()[::-1][:6]
            pairs = []
            for idx in top_idxs:
                i = idx // cm_no_diag.shape[1]
                j = idx % cm_no_diag.shape[1]
                if cm_no_diag[i, j] == 0:
                    continue
                pairs.append((i, j, int(cm_no_diag[i, j])))
            # save small text file with examples
            with open(OUT_DIR / f"{name}_misclassified_examples.txt", "w", encoding="utf8") as f:
                f.write(f"Top confusion pairs for model {name}\n\n")
                for (i,j,count) in pairs:
                    true_lab = classes[i]
                    pred_lab = classes[j]
                    f.write(f"PAIR: true={true_lab} predicted={pred_lab} count={count}\n")
                    # find indices in test set corresponding to this pair
                    sel = [k for k, (t,p) in enumerate(zip(y_test, y_pred)) if (t==true_lab and p==pred_lab)]
                    # write up to 10 examples from these
                    for sidx in sel[:10]:
                        doc_idx = idx_test[sidx]
                        text = "[original text not available]"
                        try:
                            df = pd.read_csv(PREP_CSV, low_memory=False)
                            text = str(df.iloc[doc_idx]["clean_text"])
                        except Exception:
                            pass
                        f.write(f"- doc_idx {doc_idx}: {text[:600]}\n")
                    f.write("\n\n")
        except Exception as e:
            print("Could not extract misclassified examples:", e)

    # Save all_results and preds_store
    with open(OUT_DIR / "phase4_all_results.json", "w", encoding="utf8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    with open(OUT_DIR / "phase4_predictions.json", "w", encoding="utf8") as f:
        json.dump(preds_store, f, ensure_ascii=False, indent=2)

    # Pairwise McNemar and bootstrap between models on same repr
    model_names = list(all_results.keys())
    comparisons = {}
    for i in range(len(model_names)):
        for j in range(i+1, len(model_names)):
            a = model_names[i]; b = model_names[j]
            if all_results[a]["repr"] != all_results[b]["repr"]:
                continue
            ya = np.array(preds_store[a]["y_pred"])
            yb = np.array(preds_store[b]["y_pred"])
            key = f"{a}_vs_{b}"
            mres = mcnemar_test(y_test, ya, yb)
            # bootstrap diff
            bres = bootstrap_accuracy_diff(y_test, ya, yb, iters=BOOTSTRAP_ITERS)
            comparisons[key] = {"mcnemar": mres, "bootstrap_acc_diff": bres}
    with open(OUT_DIR / "phase4_model_comparisons.json", "w", encoding="utf8") as f:
        json.dump(comparisons, f, ensure_ascii=False, indent=2)

    elapsed = time.time() - start_time
    print("Phase 4 completed in %.1f seconds. Outputs in %s" % (elapsed, OUT_DIR))

if __name__ == "__main__":
    main()


Documents: 27578 Classes: ['Books' 'Clothing & Accessories' 'Electronics' 'Household']
Train size: 22062 Test size: 5516
Found model files: [WindowsPath('phase3_outputs/TFIDF_LinearSVC_model.pkl'), WindowsPath('phase3_outputs/TFIDF_LogisticRegression_model.pkl'), WindowsPath('phase3_outputs/TFIDF_RandomForest_model.pkl'), WindowsPath('phase3_outputs/W2V_mean_LinearSVC_model.pkl'), WindowsPath('phase3_outputs/W2V_mean_LogisticRegression_model.pkl'), WindowsPath('phase3_outputs/W2V_mean_RandomForest_model.pkl')]

Processing model file: phase3_outputs\TFIDF_LinearSVC_model.pkl -> name: TFIDF_LinearSVC_model

Processing model file: phase3_outputs\TFIDF_LogisticRegression_model.pkl -> name: TFIDF_LogisticRegression_model

Processing model file: phase3_outputs\TFIDF_RandomForest_model.pkl -> name: TFIDF_RandomForest_model

Processing model file: phase3_outputs\W2V_mean_LinearSVC_model.pkl -> name: W2V_mean_LinearSVC_model

Processing model file: phase3_outputs\W2V_mean_LogisticRegression_mod